<a href="https://colab.research.google.com/github/eshernan/big-data-postgraduate/blob/main/supersalud/ProfesorSesion3Supersalud_BigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cuidado, esta es la copia del Profesor

## Pipelines y automatizacion  

### Flujo del Pipeline

- Lectura de parámetros desde config.json

    - Contendrá:

      - URLs de los datasets (2023–2024, 1er y 2do semestre).
      - Correo de notificación.
      - Umbrales de validación (ej. 30% nulos, 20% desbalance de género, etc
- Descarga y carga de los CSV:
    - Validar separador de campos (sep=";" o sep=",", algunos de estos datasets usan ;).
      - Homogeneizar nombres y tipos de columnas.
    - Validación de estructura
      - Confirmar que todos los datasets tengan los mismos campos.
      - Conversión de tipos sugeridos (int, float, category, datetime
    - Conversión a Parquet
      - Escribir en formato Parquet, particionado por periodo y mes.
- Validaciones con reglas de negocio
    - Nulos en pet_cod_depto o pet_cod_mpio: si >30% ⇒ cancelar pipeline y notificar.
    - Edad (afec_edad): si >30% >90 años ⇒ notificar + crear tabla dinámica (10 grupos etarios).
    - Género (afec_genero): si desbalance Hombre/Mujer >20% ⇒ notificar.
    - Nulos en afec_cod_depto: si >5% ⇒ notificar.
  - Notificación por correo
    - En Colab se puede usar smtplib para enviar correos, o bien integrarlo con un servicio externo (ej. Gmail API).

### Frameworks
 Cuando se trata de automatizar y orquestar flujos de trabajo, existen herramientas avanzadas como

 - Apache Airflow: https://airflow.apache.org/
 - Luigi: https://luigi.readthedocs.io/en/stable/#
 - Kedro: https://kedro.org/

 Cada una de las alternativas en muchos casos requiere instalaciones específicas y dadas las limitaciones de nuestras plataforma (Google Colab), usaremos un esquema custom (Crearemos nuestro propio Pipeline).

 Otras alternativas mucho más empresariales, que corren sobre Java son:
 - Apache NIFI: https://nifi.apache.org/
 - AWS Glue: https://aws.amazon.com/es/glue/
 - Confluent: https://www.confluent.io/
 - Azure DataFactory:  https://learn.microsoft.com/en-us/fabric/data-factory/
 - Google Dataflow: https://cloud.google.com/products/dataflow    





In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

## Crear el archivo de parametrización

In [ ]:

# Definir el diccionario de configuración
config = {
    "urls": {
        "2023S2": "https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data",
        "2024S1": "https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data",
        "2024S2": "https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data"
    },
    "notify": {
        "email_to": "alertas-datos@supersalud.gov.co",
        "email_from": "pipelines@supersalud.gov.co",
        "smtp_host": "smtp.gmail.com",
        "smtp_port": 587,
        "use_tls": True
    },
    "thresholds": {
        "null_pet_cod_depto_or_mpio_cancel": 0.30,   # 30%
        "age_over_90_alert": 0.30,                   # 30%
        "gender_imbalance_alert": 0.20,              # 20%
        "null_afec_cod_depto_alert": 0.05            # 5%
    },
    "storage": {
        "base_dir": "parquet_output",
        "format": "parquet",
        "compression": "snappy",
        "partitions": ["periodo", "mes"]
    },
    "age_groups": [2,5,11,17,26,35,45,59,74],
    "mappings": {
        "genero": {
            "M":"Hombre", "H":"Hombre", "Masculino":"Hombre", "Hombre":"Hombre",
            "F":"Mujer",  "Femenino":"Mujer", "Mujer":"Mujer"
        }
    }
}

# Guardar como archivo JSON
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

print("Archivo config.json creado correctamente")

Archivo config.json creado correctamente


## Uso de Polars, para mejorar desempeño

In [ ]:
!pip -q install polars pyarrow requests

In [ ]:
import polars as pl
import requests, io, json

## Validaciones de las URL y los separadores de campo en los csv

- Se verifica que se puedan leer las URL
- Se verifica que se pueda leer el CSV con un separador de campo, dentro de la lista
- Se verifica que la URL leida se pueda leer con el separador de campos definido

In [ ]:
# Función (tarea), para leer los archivos de datos, que fueron colocados en el json de configuración

def read_csv_auto(bytes_content: bytes) -> pl.DataFrame:
  print("Iniciando")
  attempts = []
  for sep in [";", ",","|"]:
      try:
          df = pl.read_csv(io.BytesIO(bytes_content), separator=sep, infer_schema_length=10000)
          attempts.append((sep, df))
      except Exception:
          continue
  if not attempts:
      raise ValueError("No se pudo leer el archivo con ; ni , ni |")
  # heurística simple: más columnas ⇒ mejor
  best = max(attempts, key=lambda x: len(x[1].columns))
  print(f"Archivo leído con el separador {best[0]}")
  return best[1]

def load_from_url(url: str) -> pl.DataFrame:
  r = requests.get(url, timeout=60, allow_redirects=True, verify=False)
  r.raise_for_status()
  return read_csv_auto(r.content)

## Validaciones y establecimiento de esquemas

In [ ]:
def assert_same_schema(dfs: list[pl.DataFrame]) -> list[str]:
    """
    Normaliza todos los nombres de columnas a minúsculas
    y valida que todos los DataFrames tengan el mismo esquema.

    Retorna la lista de columnas base si todo es consistente.
    """
    # Paso 1: normalizar nombres a minúscula en todos los DataFrames
    dfs_normalized = [df.rename({c: c.lower() for c in df.columns}) for df in dfs]

    # Paso 2: validar esquemas
    colsets = [set(df.columns) for df in dfs_normalized]
    base = colsets[0]
    if not all(cols == base for cols in colsets[1:]):
        diffs = []
        for i, cols in enumerate(colsets):
            missing = base - cols
            extra = cols - base
            if missing:
                diffs.append(f"DF[{i}] le faltan columnas: {missing}")
            if extra:
                diffs.append(f"DF[{i}] tiene columnas extra: {extra}")
        raise ValueError("Esquema distinto entre archivos:\n" + "\n".join(diffs))

    return list(dfs_normalized[0].columns)



## Definición de Typos de datos

In [ ]:
def coerce_types(df: pl.DataFrame) -> pd.DataFrame:
    col_int_nullable = [
        "pet_cod_depto", "pet_cod_mpio", "afec_cod_depto", "afec_cod_mpio",
        "ent_cod_depto", "ent_cod_mpio", "ent_cod_sns",
        "cod_macromot", "cod_motgen", "cod_motesp", "cod_tipo_motesp", "cod_subtipo_motesp",
        "mes", "afec_edad"
    ]
    for c in col_int_nullable:
        if c in df.columns:
            df = df.with_columns(pl.col(c).cast(pl.Int64, strict=False))
    if "periodo" in df.columns:
        df = df.with_columns(pl.col("periodo").cast(pl.Utf8, strict=False))
    # Normalización de género
    if "afec_genero" in df.columns:
        mapping = { "M":"Hombre","H":"Hombre","Masculino":"Hombre","Hombre":"Hombre",
                    "F":"Mujer","Femenino":"Mujer","Mujer":"Mujer" }
        df = df.with_columns(
            pl.col("afec_genero").cast(pl.Utf8, strict=False).str.strip().map_dict(mapping, default=None).alias("afec_genero")
        )
    return df


## Escritura en Parquet particionado por periodo y mes

Parquet comprime, acelera lecturas y estandariza tipos.

Particionar por **periodo** y **mes** permite cortes temporales rápidos (ej., vigilancia mensual, comparativos semestrales).

## Esquema de particionamiento sin formato Hive, que no incluye el nombre del campo en el folder

In [ ]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import os

def write_partitioned_parquet(df, base_dir, partitions, compression="snappy"):
    """
    Escribe un DataFrame de Polars a Parquet particionado usando PyArrow Dataset API.
    """
    os.makedirs(base_dir, exist_ok=True)

    # convertir Polars a Arrow
    table = df.to_arrow()

    # Configurar opciones de escritura de Parquet
    pq_options = pq.ParquetWriter  # solo para dejar claro, no es necesario instanciar aquí

    ds.write_dataset(
        data=table,
        base_dir=base_dir,
        partitioning=partitions,
        format="parquet",
        existing_data_behavior="overwrite_or_ignore"
    )


## Particionamiento con esquema de Hive, para incluir el campo en el nombre de la partición

In [ ]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import os

def write_partitioned_parquet(df, base_dir, partitions, compression="snappy"):
    """
    Escribe un DataFrame de Polars a Parquet particionado usando PyArrow Dataset API.
    """
    os.makedirs(base_dir, exist_ok=True)

    # convertir Polars a Arrow
    table = df.to_arrow()

    # 2) Construir schema de particiones y activar flavor "hive"
    part_fields = [table.schema.field(name) for name in partitions]
    part_schema = pa.schema(part_fields)
    hive_part = ds.partitioning(part_schema, flavor="hive")

    ds.write_dataset(
        data=table,
        base_dir=base_dir,
        partitioning=hive_part,
        format="parquet",
        existing_data_behavior="overwrite_or_ignore"
    )



## Generación de Reglas
### Reglas de terminación anticipada

Si **pet_cod_depto** o **pet_cod_mpio tienen** > 30% nulos (umbral en config.json) ⇒ cancelar pipeline y enviar correo.

**Justificación:** la capacidad de territorializar reclamos es clave para vigilancia diferencial y alertas regionales; sin esa georreferenciación mínima, la analítica posterior pierde validez.

# Reglas de Negocio post-procesamiento

- Regla porcentaje de edad
- Regla de genero
- Regla de nulos en código departamento para afectado

In [ ]:
def null_rate(series: pl.Series) -> float:
    return float(series.null_count()) / len(series)

def check_cancel_location_nulls(df: pl.DataFrame, th: float) -> tuple[bool, str]:
    rates = {}
    for c in ["pet_cod_depto","pet_cod_mpio"]:
        if c in df.columns:
            rates[c] = null_rate(df[c])
    max_rate = max(rates.values()) if rates else 0.0
    cancel = max_rate > th
    msg = f"Nulos pet_cod_depto={rates.get('pet_cod_depto',0):.2%}, pet_cod_mpio={rates.get('pet_cod_mpio',0):.2%}"
    return cancel, msg


In [ ]:
def age_over_90_rate(df: pl.DataFrame) -> float:
    if "afec_edad" not in df.columns:
        return 0.0
    valid = df.filter(pl.col("afec_edad").is_not_null())
    if valid.height == 0:
        return 0.0
    return float(valid.filter(pl.col("afec_edad") > 90).height) / valid.height


## Generación de grupos etáreos

In [ ]:
def add_age_group(df: pl.DataFrame, cutpoints: list[int]) -> pl.DataFrame:
    # cutpoints: límites superiores intermedios (ej. [2,5,11,17,26,35,45,59,74])
    def label(age):
        if age is None: return None
        bounds = [2,5,11,17,26,35,45,59,74]
        labels = ["0-2","3-5","6-11","12-17","18-26","27-35","36-45","46-59","60-74","75+"]
        for i, b in enumerate(bounds):
            if age <= b:
                return labels[i]
        return "75+"
    return df.with_columns(
        pl.col("afec_edad").map_elements(label, return_dtype=pl.Utf8).alias("grupo_edad")
    )

# Tabla dinámica para examinar edades

def pivot_age(df: pl.DataFrame) -> pl.DataFrame:
    # distribución por grupo de edad (total)
    tmp = df.filter(pl.col("grupo_edad").is_not_null()) \
            .group_by("grupo_edad").agg(pl.len().alias("n")) \
            .sort("grupo_edad")
    # porcentaje
    total = tmp["n"].sum()
    return tmp.with_columns((pl.col("n")/total).alias("pct"))


##Regla de balance de género

In [ ]:
def gender_imbalance(df: pl.DataFrame) -> float:
    if "afec_genero" not in df.columns:
        return 0.0
    g = df.filter(pl.col("afec_genero").is_in(["Mujer","Hombre"])) \
          .group_by("afec_genero").agg(pl.len().alias("n"))
    if g.height < 2:
        return 0.0
    total = g["n"].sum()
    p_m = float(g.filter(pl.col("afec_genero")=="Mujer")["n"][0]) / total
    p_h = float(g.filter(pl.col("afec_genero")=="Hombre")["n"][0]) / total
    return abs(p_m - p_h)  # diferencia absoluta


In [ ]:
def suma(sumando1,sumando2):
  return sumando1 + sumando2


def resta(minuendo, sustraendo):
  return minuendo - sustraendo

def division(dividendo, divisor):
  if divisor == 0:
    raise ValueError("No se puede dividir por cero")
  return dividendo / divisor



# Generación del pipelina


In [ ]:
import os, json

# 1) Cargar config
with open("config.json","r") as f:
    CFG = json.load(f)

urls = CFG["urls"]
thr  = CFG["thresholds"]
noti = CFG["notify"]
store= CFG["storage"]

In [ ]:
for name, url in urls.items():
    print("url {1} en archivo".format(name, url))

url https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data en archivo
url https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data en archivo
url https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data en archivo


In [ ]:

# 2) Descargar / leer datasets
dfs = []
for name, url in urls.items():
    df = load_from_url(url)
    print("Descargando archivo {1}".format(name, url))
    # aseguramos columnas periodo/mes presentes; si no, inferir de archivo o agregar
    if "periodo" not in df.columns:
        df = df.with_columns(pl.lit(name).alias("periodo"))  # p.ej., "2024S2"
    if "mes" not in df.columns:
        # si el archivo no trae "mes", pero el dataset es semestral, podríamos derivar de una columna de fecha si existe.
        # En demo, dejamos null y permitimos escribir; ideal: inferir.
        df = df.with_columns(pl.lit(None).cast(pl.Int64).alias("mes"))
    #df = coerce_types(df)
    dfs.append(df)

Iniciando
Archivo leído con el separador ;
Descargando archivo https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data
Iniciando
Archivo leído con el separador ,
Descargando archivo https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data
Iniciando
Archivo leído con el separador ;
Descargando archivo https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data


In [ ]:
for i, df in enumerate(dfs):
    print(f"Headers of DataFrame {i}:")
    print(df.columns)
    print("-" * 20)

Headers of DataFrame 0:
['OBJECTID', 'periodo', 'mes', 'pqr_canal', 'pet_cod_depto', 'pet_cod_mpio', 'id_afec', 'afec_parentesco', 'afec_genero', 'afec_edad', 'afec_edadr', 'afec_educ', 'afec_regafiliacion', 'afec_getnico', 'afec_pobespecial', 'afec_cod_depto', 'afec_cod_mpio', 'ent_nombre', 'ent_tipovig_sns', 'ent_cod_sns', 'ent_alias_sn', 'ent_cod_depto', 'ent_cod_mpio', 'cod_macromot', 'macromotivo', 'cod_motgen', 'motivo_general', 'cod_motesp', 'motivo_especifico', 'cod_tipo_motesp', 'tipo_de_motivo_especifico', 'cod_subtipo_motesp', 'subtipo_de_motivo_especifico', 'patologia_1', 'patologia_tipo', 'cie_10', 'alto_costo', 'clasificacion_de_riesgo']
--------------------
Headers of DataFrame 1:
['OBJECTID', 'periodo', 'mes', 'pqr_canal', 'pet_cod_depto', 'pet_cod_mpio', 'id_afec', 'afec_parentesco', 'afec_genero', 'afec_edad', 'afec_edadr', 'afec_educ', 'afec_regafiliacion', 'afec_getnico', 'afec_pobespecial', 'afec_cod_depto', 'afec_cod_mpio', 'ent_nombre', 'ent_tipovig_sns', 'ent_co

In [ ]:

# 3) Validar esquema
_ = assert_same_schema(dfs)

In [ ]:
# 4) Unificar
#df_all = pl.concat(dfs, how="vertical_relaxed")
# Read Parquet files from the partitioned directory
df_all = pl.read_parquet("parquet_output/**/**/*.parquet", hive_partitioning=True)

# Display the first few rows to verify
display(df_all.head())

OBJECTID,pqr_canal,pet_cod_depto,pet_cod_mpio,id_afec,afec_parentesco,afec_genero,afec_edad,afec_edadr,afec_educ,afec_regafiliacion,afec_getnico,afec_pobespecial,afec_cod_depto,afec_cod_mpio,ent_nombre,ent_tipovig_sns,ent_cod_sns,ent_alias_sn,ent_cod_depto,ent_cod_mpio,cod_macromot,macromotivo,cod_motgen,motivo_general,cod_motesp,motivo_especifico,cod_tipo_motesp,tipo_de_motivo_especifico,cod_subtipo_motesp,subtipo_de_motivo_especifico,patologia_1,patologia_tipo,cie_10,alto_costo,clasificacion_de_riesgo,periodo,mes
i64,str,i64,i64,str,str,str,i64,str,str,str,str,str,i64,i64,str,str,str,str,i64,i64,i64,str,i64,str,i64,str,i64,str,i64,str,str,str,str,str,str,i64,i64
374910,"""Web""",68,68001,"""US- 000003458211""","""A nombre propio""","""MUJER""",53,"""DE 50 A 62 AÑOS""","""No Aplica""","""Subsidiado""","""No Aplica""","""No Aplica""",68,68001,"""NUEVA EPS""","""RÉGIMEN SUBSIDIADO""","""EPSS41""","""NUEVA EPS""",68,68001,1,"""BARRERAS EN EL ACCESO A TECNOL…",102,"""NEGACIÓN DE TECNOLOGÍAS EN SAL…",10203,"""NEGACIÓN EN LA ATENCIÓN EN OTR…",1020308,"""PROCEDIMIENTOS NO QUIRÚRGICOS""",1020308001,"""IMAGENOLOGÍA""","""ENFERMEDADES CRÓNICAS NO TRASM…","""HTA (HIPERTENSIÓN ARTERIAL)""","""TRASTORNO ARTICULAR, NO ESPECI…","""No aplica""","""SIMPLE""",2023,10
374911,"""Telefónico""",5,5001,"""US- 000003458212""","""Madre""","""MUJER""",9,"""DE 6 A 12 AÑOS""","""Primaria Incompleta""","""Contributivo""","""No Aplica""","""No Aplica""",5,5001,"""COMPENSAR""","""RÉGIMEN CONTRIBUTIVO""","""EPS008""","""COMPENSAR""",5,5001,4,"""INSATISFACCIÓN DEL USUARIO CON…",404,"""INSATISFACCIÓN RELACIONADA CON…",40403,"""INCONSISTENCIAS EN LA AFILIACI…",4040309,"""OTRAS INCONSISTENCIAS""",null,null,"""PROBLEMAS RELACIONADOS CON FAC…","""PROBLEMAS RELACIONADOS CON FAC…","""PROBLEMAS RELACIONADOS CON FAC…","""No aplica""","""SIMPLE""",2023,10
374912,"""Escrito""",11,11001,"""US- 000003458213""","""Otro""","""MUJER""",20,"""DE 18 A 24 AÑOS""","""No Aplica""","""Subsidiado""","""No Aplica""","""No Aplica""",8,8758,"""SURAMERICANA EPS""","""RÉGIMEN CONTRIBUTIVO""","""EPS010""","""EPS SURA""",8,8758,1,"""BARRERAS EN EL ACCESO A TECNOL…",101,"""FALTA DE OPORTUNIDAD EN LA PRE…",10103,"""FALTA DE OPORTUNIDAD EN LA ATE…",1010308,"""PROCEDIMIENTOS NO QUIRÚRGICOS""",1010308006,"""OTROS PROCEDIMIENTOS NO QUIRÚR…","""ENFERMEDADES OSTEOARTICULARES""","""NEUROLÓGICAS""","""CEFALEA""","""No aplica""","""SIMPLE""",2023,10
374913,"""Web""",76,76001,"""US- 000002503040""","""Otro""","""MUJER""",67,"""MAYOR DE 63 AÑOS ""","""No Aplica""","""Contributivo""","""No Aplica""","""No Aplica""",76,76001,"""SURAMERICANA EPS""","""RÉGIMEN CONTRIBUTIVO""","""EPS010""","""EPS SURA""",76,76001,1,"""BARRERAS EN EL ACCESO A TECNOL…",103,"""FALTA DE OPORTUNIDAD EN LA AUT…",10302,"""FALTA DE OPORTUNIDAD EN LA AUT…",1030203,"""ESPECIALIDADES MÉDICAS """,1030203080,"""OTORRINOLARINGOLOGÍA""","""PROBLEMAS RELACIONADOS CON FAC…","""PROBLEMAS RELACIONADOS CON FAC…","""OTROS PROBLEMAS RELACIONADOS C…","""No aplica""","""SIMPLE""",2023,10
374914,"""Telefónico""",47,47189,"""US- 000003458214""","""A nombre propio""","""MUJER""",38,"""DE 38 A 49 AÑOS""","""Secundaria""","""Subsidiado""","""No Aplica""","""No Aplica""",47,47189,"""MUTUAL SER""","""RÉGIMEN SUBSIDIADO""","""ESS207""","""MUTUAL SER""",47,47189,1,"""BARRERAS EN EL ACCESO A TECNOL…",101,"""FALTA DE OPORTUNIDAD EN LA PRE…",10104,"""FALTA DE OPORTUNIDAD EN LA ENT…",1010401,"""MEDICAMENTOS UPC""",null,null,"""PROBLEMAS RELACIONADOS CON FAC…","""PROBLEMAS RELACIONADOS CON FAC…","""DERMATITIS ATÓPICA, NO ESPECIF…","""No aplica""","""SIMPLE""",2023,10


In [ ]:

# 6) Escribir Parquet particionado
os.makedirs(store["base_dir"], exist_ok=True)
write_partitioned_parquet(df_all, store["base_dir"], store["partitions"], compression=store["compression"])

In [ ]:
!du -sh parquet_output

99M	parquet_output


## Ejercicios hasta esta linea

In [ ]:

# 5) REGLA GATING: nulos en pet_cod_depto / pet_cod_mpio
cancel, msg = check_cancel_location_nulls(df_all, thr["null_pet_cod_depto_or_mpio_cancel"])
if cancel:
    send_email(
        noti["smtp_host"], noti["smtp_port"], noti["use_tls"],
        noti["email_from"], noti["email_to"],
        "[Pipeline CANCELADO] Nulos en ubicación del peticionario",
        f"{msg}\nUmbral: {thr['null_pet_cod_depto_or_mpio_cancel']:.0%}"
    )
    raise SystemExit("Pipeline cancelado por nulos en ubicación del peticionario.")


In [ ]:
# Regla de validaciones nulos por columna  6
def null_rate_col(df: pl.DataFrame, col: str) -> float:
    return null_rate(df[col]) if col in df.columns else 0.0

In [ ]:

# 7) Reglas post-carga (alertas)
rate_age90 = age_over_90_rate(df_all)
if rate_age90 > thr["age_over_90_alert"]:
    print("Regla de Edad 20% mayor a 90 años aplicada")
    send_email(
        noti["smtp_host"], noti["smtp_port"], noti["use_tls"],
        noti["email_from"], noti["email_to"],
        "[Alerta] Tasa de edades >90 supera umbral",
        f"Tasa >90: {rate_age90:.2%} (Umbral {thr['age_over_90_alert']:.0%})"
    )
    # tabla dinámica
    df_age = add_age_group(df_all, CFG["age_groups"])
    tabla = pivot_age(df_age)
    print(tabla)  # mostrar en clase
else:
    print("Regla de Edad 20% mayor a 90 años NO aplicada")

imbalance = gender_imbalance(df_all)
if imbalance > thr["gender_imbalance_alert"]:
    print("Regla de Género aplicada")
    send_email(
        noti["smtp_host"], noti["smtp_port"], noti["use_tls"],
        noti["email_from"], noti["email_to"],
        "[Alerta] Desbalance de género",
        f"Diferencia absoluta Mujer vs Hombre: {imbalance:.2%} (Umbral {thr['gender_imbalance_alert']:.0%})"
    )
else:
    print("Regla de Género NO aplicada")

rate_afec_depto = null_rate_col(df_all, "afec_cod_depto")
if rate_afec_depto >= thr["null_afec_cod_depto_alert"]:
    print("Regla de nulos en afec_cod_depto aplicada")
    send_email(
        noti["smtp_host"], noti["smtp_port"], noti["use_tls"],
        noti["email_from"], noti["email_to"],
        "[Alerta] Nulos en afec_cod_depto",
        f"Tasa de nulos: {rate_afec_depto:.2%} (Umbral {thr['null_afec_cod_depto_alert']:.0%})"
    )
else:
    print("Regla de nulos en afec_cod_depto NO aplicada")

print("✅ Pipeline completado (sin alertas).")

Regla de Edad 20% mayor a 90 años NO aplicada
Regla de Género NO aplicada
Regla de nulos en afec_cod_depto NO aplicada
✅ Pipeline completado (sin alertas).
